# 03b - RM-b: frozen encoder + classification head

Encoder IndoBERT dibekukan dan hanya classification head yang dilatih. Embedding
mean-pool diekstrak sekali lalu dipakai ulang oleh seluruh konfigurasi head,
sehingga satu epoch hanya berupa perkalian matriks kecil.

Prasyarat: `02_preprocessing.ipynb` sudah dijalankan.

In [ ]:
from src.config import settings
from src.services.campaign import CampaignRunner

OUT_DIR = settings.output_dir / "baseline"

runner = CampaignRunner(out_dir=OUT_DIR)
print("device        :", runner.device)
print("encoder       :", runner.model_name)
print("keluaran      :", runner.out_dir)
print("train/val/test:", [len(frame) for frame in runner.data.frames.values()])

## 1. Ekstraksi fitur beku

In [ ]:
features = runner.features

print(f"dari cache      : {features.from_cache}")
print(f"dimensi         : {features.hidden_dim}")
print(f"waktu ekstraksi : {features.extract_time_s:.2f} s")
print(f"peak GPU memory : {features.extract_peak_mem_mb:.0f} MB")
for split in ("train", "val", "test"):
    embeddings, labels = features[split]
    print(f"  {split:5s}: {embeddings.shape} | label {labels.shape}")

Ekstraksi berjalan sekali per encoder lalu di-cache ke
`outputs/baseline/features/<nama_encoder>/`. Biaya ini tetap dihitung sebagai
bagian waktu latih RM-b agar perbandingannya dengan RM-a jujur, walau
diamortisasi ke seluruh konfigurasi head.

## 2. Konfigurasi

In [ ]:
from src.models.schemas import RMBConfig

config = RMBConfig()
print(config.model_dump())

Default mengikuti Peters dkk. (2019) untuk transfer berbasis fitur: head
diinisialisasi acak sehingga butuh learning rate lebih tinggi daripada RM-a.

## 3. Jalankan

In [ ]:
row = runner.run(
    "rmb",
    config.model_dump(),
    note="baseline head linear di atas fitur beku",
)

print(f"run #{row['run_id']}")
print(f"  val F1-macro     : {row['val_f1_macro']:.4f}")
print(f"  val F1 judi      : {row['val_f1_judi']:.4f}")
print(f"  epoch terbaik    : {row['best_epoch']} dari {row['epochs']}")
print(f"  waktu latih head : {row['head_train_time_s']:.2f} s")
print(f"  + ekstraksi      : {row['extract_time_s']:.2f} s")
print(f"  total waktu latih: {row['train_time_s']:.2f} s")
print(f"  trainable params : {row['trainable_params']:,}")

## 4. Bandingkan head linear dan MLP

In [ ]:
row_mlp = runner.run(
    "rmb",
    {"head_arch": "mlp", "hidden_dim": 256},
    note="uji apakah satu hidden layer menutup selisih terhadap RM-a",
)

print(f"linear: F1-macro {row['val_f1_macro']:.4f} | "
      f"{row['trainable_params']:,} params")
print(f"mlp   : F1-macro {row_mlp['val_f1_macro']:.4f} | "
      f"{row_mlp['trainable_params']:,} params")
print(f"selisih: {(row_mlp['val_f1_macro'] - row['val_f1_macro']) * 100:+.2f} pp")

## Ringkasan

Perbandingan trainable parameter terhadap RM-a adalah inti klaim efisiensi
skenario ini. Pencarian kapasitas head yang optimal dilakukan di
`04_tuning_campaign.ipynb`.

Lanjut ke `03c_rmc_rac.ipynb`.